# JEPA-TTS Supercomputer Pipeline
This notebook automates the entire pipeline: downloading the fixed codebase, installing dependencies, training from scratch, and running inference/testing on the supercomputer.

### 1. Clone Codebase & Checkout Prototype Branch

In [ ]:
!git clone https://github.com/OmarA32/Audio-JEPA-Arabic-TTS.git
%cd Audio-JEPA-Arabic-TTS
!git checkout prototype/v3.0.0


### 2. Install Dependencies
Installs all libraries directly into the notebook kernel (including `vocos`).

In [ ]:
!pip install -r requirements.txt

### 3. Clear Old Weights (Safety)

In [ ]:
!rm -rf training_logs

### 3.5 Download Arabic Datasets
Downloads Nawar Halabi's high-quality single-speaker dataset to be used for fine-tuning. Common Voice downloads automatically via HuggingFace during training.

In [ ]:
!mkdir -p data
!wget -nc https://en.arabicspeechcorpus.com/arabic-speech-corpus.zip -O data/arabic-speech-corpus.zip
!unzip -q -o data/arabic-speech-corpus.zip -d data/arabic-speech-corpus
!rm data/arabic-speech-corpus.zip

### 3.6 Download English Datasets
Downloads LJSpeech (single speaker) and LibriTTS (multi-speaker) via torchaudio.

In [ ]:
import torchaudio
torchaudio.datasets.LJSPEECH("./data", download=True)
torchaudio.datasets.LIBRITTS("./data", url="train-clean-100", download=True)

### 4A. Run Pre-Training
Train on Common Voice Arabic.

In [ ]:
!python train.py --lang arabic --db common_voice

### 4B. Run Fine-Tuning
Fine tune on Nawar Halabi (Arabic) or LJSpeech (English).

In [ ]:
!python train.py --lang arabic --db nawar_halabi --resume

### 6. Test Vocoder (Ground Truth Quality)
If you want to test the raw quality of the vocoder against the dataset (without the neural network's influence), run this test.

In [ ]:
!python test_vocoder_ground_truth.py --vocoder vocos

### 7. Test TTS model with new text.
You can pass any custom text to generate here:

In [ ]:
!python inference.py --lang arabic --db nawar_halabi --text "أي نص عربي تريد" --output "my_custom_audio.wav"

### 8. Upload Weights to Hugging Face
Since supercomputers delete data when shut down, run this cell to push your saved epochs (100, 200, etc.) to your HF account. 
First, get an Access Token from your Hugging Face settings (make sure it is a **WRITE** token).

In [ ]:
import os
from huggingface_hub import HfApi, login

# 1. Login (Paste your token below)
hf_token = "YOUR_HUGGINGFACE_WRITE_TOKEN" 
login(token=hf_token)

# 2. Push the entire training_logs folder to a new or existing repository
repo_id = "your-username/JEPA-Arabic-TTS-Checkpoints" # Change this!

api = HfApi()
api.create_repo(repo_id=repo_id, exist_ok=True)
api.upload_folder(
    folder_path="training_logs",
    repo_id=repo_id,
    repo_type="model"
)
print("Upload complete! All epochs are now safely stored in Hugging Face.")